# 03 · QLoRA on Mistral-7B (PEFT)

Fine-tune a **7B** model on a single ~16GB GPU using **QLoRA** (4-bit).
**Requires a CUDA GPU** — use Google Colab (T4) or a cloud GPU. Will not run on CPU/Mac.
Theory: [docs/05_qlora_explained.md](../docs/05_qlora_explained.md).


## 1. Install (GPU runtime)


In [ ]:
!pip install -q transformers datasets accelerate peft trl bitsandbytes


## 2. Confirm a GPU is present


In [ ]:
import torch; assert torch.cuda.is_available(), 'Switch to a GPU runtime!'
print(torch.cuda.get_device_name(0))


## 3. Load Mistral-7B in 4-bit (NF4 + double quant)
This is the heart of QLoRA: the frozen base lives in 4-bit, compute happens in bf16.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = 'mistralai/Mistral-7B-v0.1'
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token; tokenizer.padding_side = 'right'
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb, device_map='auto')
model.config.use_cache = False


## 4. Prepare for k-bit training + attach LoRA


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    task_type='CAUSAL_LM')
model = get_peft_model(model, lora)
model.print_trainable_parameters()


## 5. Data + train
Same Alpaca formatting as notebook 02. For real runs use a larger dataset (e.g. `mlabonne/guanaco-llama2-1k`).


In [ ]:
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer

ds = load_dataset('json', data_files='../data/sample_instructions.jsonl', split='train')
EOS = tokenizer.eos_token
def to_text(ex):
    inp = f"\n\n### Input:\n{ex['input']}" if ex['input'] else ''
    return {'text': f"### Instruction:\n{ex['instruction']}{inp}\n\n### Response:\n{ex['output']}" + EOS}
ds = ds.map(to_text, remove_columns=ds.column_names)

args = TrainingArguments(output_dir='out_qlora', num_train_epochs=1,
    per_device_train_batch_size=2, gradient_accumulation_steps=8, learning_rate=2e-4,
    lr_scheduler_type='cosine', warmup_ratio=0.03, logging_steps=5,
    bf16=True, optim='paged_adamw_8bit', gradient_checkpointing=True, report_to='none')
trainer = SFTTrainer(model=model, args=args, train_dataset=ds,
    dataset_text_field='text', max_seq_length=1024, tokenizer=tokenizer)
trainer.train()
trainer.save_model('qlora-adapter')


## 6. Merge + save (optional, for deployment)
Reload base in 16-bit, merge the adapter, and you have a standalone model for FastAPI/Ollama.


In [ ]:
# See src/merge_adapter.py for the full script. Sketch:
# from peft import PeftModel
# base = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, device_map='auto')
# merged = PeftModel.from_pretrained(base, 'qlora-adapter').merge_and_unload()
# merged.save_pretrained('merged_model'); tokenizer.save_pretrained('merged_model')


## Takeaways
- QLoRA put a 7B model + training into ~16GB by freezing the base in 4-bit.
- Only the LoRA adapters trained, in bf16, with a paged optimizer to avoid OOM.
- Next: [04_finetune_llama2.ipynb](04_finetune_llama2.ipynb) for an instruction-tuning workflow.
